# Base de vendas - AWS

In [0]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

## Leitura da base em spark e pandas

In [0]:
# spark_aws = spark.table('workspace.default.aws')
# pandas_aws = spark_aws.toPandas()
pandas_aws = pd.read_csv('SaaS-Sales.csv')

In [0]:
display(pandas_aws)

## Investigação e tratamento

padronização do nome de colunas
 - letras minúsculas
 - separação por underline `_`

In [0]:
pandas_aws.columns = [colname.lower().replace(' ', '_') for colname in pandas_aws.columns]
pandas_aws.columns

arredondando decimais para 2 casas

In [0]:
pandas_aws[['sales', 'profit']] = pandas_aws[['sales', 'profit']].round(2)
pandas_aws['order_date'] = pd.to_datetime(
    pandas_aws['order_date']
)
display(pandas_aws.head())

verificação de valores nulos

In [0]:

null_rows = pandas_aws[pandas_aws.isnull().any(axis=1)]
if not null_rows.empty:
    display(null_rows)
else:
    print('Não há valores nulos')

verificação de chaves do pedido duplicadas

In [0]:
duplicate_orders = pandas_aws['order_id'].value_counts()
if duplicate_orders.max() > 1:
    print('Há pedidos duplicados')
    display(pandas_aws[pandas_aws['order_id'].isin(duplicate_orders[duplicate_orders > 1].index)])
else:
    print('Não há pedidos duplicados')

verificação de mesmo pedido, mas destinos / clientes diferentes

In [0]:
duplicates = (
    pandas_aws.groupby('order_id')
    .agg({
        'row_id'     :    "count",
        'customer'   :    "nunique",
        'country'    :    "nunique",
    })
    .rename(columns={'row_id': 'count', 'customer': 'distinct_customers', 'country': 'distinct_countries'})
)
filtered_duplicates = duplicates[(duplicates['distinct_customers'] > 1) | (duplicates['distinct_countries'] > 1)]
display(pandas_aws[pandas_aws['order_id'].isin(filtered_duplicates.index)])

verificação de sales ou quantity menor que 1

In [0]:
negatives = (pandas_aws['sales'] <= 0) | (pandas_aws['quantity'] <= 0)
if negatives.any():
    print('Há valores negativos (ou 0)')
    display(pandas_aws[negatives])
else:
    print('Não há valores negativos (ou 0)')

### Novos campos calculados

In [0]:
pandas_aws['cost'] = round((pandas_aws['sales'] - pandas_aws['profit']), 2)
pandas_aws['product_cost'] = round((pandas_aws['cost'] / pandas_aws['quantity']), 2)
pandas_aws['profit_margin'] = round((pandas_aws['profit'] / pandas_aws['sales'] * 100), 2)
display(pandas_aws.head())

### Construindo tabela agregada por carrinho

In [0]:
aws_carrinho = (
    pandas_aws.groupby(['order_id'])
        .agg({
            'order_date'     :   "last",
            'date_key'       :   "last",
            'contact_name'   :   "last",
            'country'        :   "last",
            'city'           :   "last",
            'region'         :   "last",
            'subregion'      :   "last",
            'customer'       :   "last",
            'customer_id'    :   "last",
            'industry'       :   "last",
            'segment'        :   "last",
            'sales'          :    'sum',
            'quantity'       :    'sum',
            'profit'         :    'sum',
            'cost'           :    'sum',
            'product_cost'   :   'mean',
            'profit_margin'  :   'mean',
        })
        .reset_index()
)
# aws_carrinho['order_date'] = pd.to_datetime(
#     aws_carrinho['order_date']
# )
display(aws_carrinho)

## Base de Clientes

In [0]:
def classificar_status(order_dates: pd.Series, reference_date: date):
    order_dates = (
        pd.to_datetime(order_dates)
            .sort_values()
            .reset_index(drop=True)
    )
    reference_date = pd.to_datetime(reference_date)
    if reference_date not in order_dates.values:
        return None

    pos = order_dates.reset_index(drop=True)[order_dates == reference_date].index[0]

    if pos == 0:
        return 'new_customer'
    elif (reference_date - order_dates.iloc[pos - 1]).days <= 30:
        return 'active_customer'
    elif pos > 1 and (order_dates.iloc[pos - 1] - order_dates.iloc[pos - 2]).days <= 30:
        return 'churned_customer'
    else:
        return 'recovery_customer'

In [0]:
aws_customers = (
    aws_carrinho.groupby(['customer_id', 'order_date'])
    .agg({
        'customer'  :  'first',
        'country'   :  'first',
        'city'      :  'first',
        'region'    :  'first',
        'subregion' :  'first',
        'industry'  :  'first',
        'segment'   :  'first',
        'sales'     :  'sum',
        'quantity'  :  'sum',
        'profit'    :  'sum',
    })
    .reset_index()
)

aws_customers['customer_status'] = aws_customers.groupby('customer_id')['order_date'].transform(
    lambda dates: [classificar_status(dates, d) for d in dates]
)

display(aws_customers)

## KPI's (calculando em nova base agregada)

In [0]:
aws_carrinho['month'] = (
    aws_carrinho['order_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

aws_carrinho_metrics = (
    aws_carrinho.groupby(['month'])
    .agg(
        total_carrinhos=('order_id', 'nunique'),
        total_consumers=('customer_id', 'nunique'),
        total_sales=('sales', 'sum'),
    )
    .reset_index()
)

aws_carrinho_metrics['ticket'] = round(
    aws_carrinho_metrics['total_sales'] / aws_carrinho_metrics['total_carrinhos'],
    2
)

aws_carrinho_metrics['spending'] = round(
    aws_carrinho_metrics['total_sales'] / aws_carrinho_metrics['total_consumers'],
    2
)

display(aws_carrinho_metrics)

In [0]:
aws_customers['month'] = (
    aws_customers['order_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

aws_customer_metrics = (
    aws_customers.groupby(['month'])
    .agg(
        total_customers=('customer_id', 'count'),
        total_churn=('customer_status', lambda x: (x == 'churned_customer').sum()),
        total_retention=('customer_status', lambda x: (x.isin(['active_customer', 'recovery_customer'])).sum())
    )
    .reset_index()
)

aws_customer_metrics['taxa_churn'] = (
    round(aws_customer_metrics['total_churn'] / aws_customer_metrics['total_customers'],2) * 100
)
aws_customer_metrics['taxa_retencao'] = (
    round(aws_customer_metrics['total_retention'] / aws_customer_metrics['total_customers'],2) * 100
)
display(aws_customer_metrics)

In [0]:
aws_metrics = (
    aws_carrinho_metrics
    .merge(
        aws_customer_metrics,
        on='month',
        how='inner'
    )
)
display(aws_metrics)

In [0]:
#meterializar aws_carrinho

spark.createDataFrame(aws_carrinho).write.mode('overwrite').saveAsTable('bronze_aws_carrinho')


spark.createDataFrame(aws_customers).write.mode('overwrite').saveAsTable('bronze_aws_customers')


spark.createDataFrame(aws_metrics).write.mode('overwrite').saveAsTable('bronze_aws_metrics')